# Hypothesis Testing in Healthcare: Drug Safety

A pharmaceutical company GlobalXYZ has just completed a randomized controlled drug trial. To promote transparency and reproducibility of the drug's outcome, they (GlobalXYZ) have presented the dataset to your organization, a non-profit that focuses primarily on drug safety.

The dataset provided contained five adverse effects, demographic data, vital signs, etc. Your organization is primarily interested in the drug's adverse reactions. It wants to know if the adverse reactions, if any, are of significant proportions. It has asked you to explore and answer some questions from the data.

The dataset `drug_safety.csv` was obtained from [Hbiostat](https://hbiostat.org/data/) courtesy of the Vanderbilt University Department of Biostatistics. It contained five adverse effects: headache, abdominal pain, dyspepsia, upper respiratory infection, chronic obstructive airway disease (COAD), demographic data, vital signs, lab measures, etc. The ratio of drug observations to placebo observations is 2 to 1.

For this project, the dataset has been modified to reflect the presence and absence of adverse effects `adverse_effects` and the number of adverse effects in a single individual `num_effects`.

The columns in the modified dataset are: 

| Column | Description |
|--------|-------------|
|`sex` | The gender of the individual |
|`age` | The age of the individual |
|`week` | The week of the drug testing |
|`trx` | The treatment (Drug) and control (Placebo) groups | 
|`wbc` | The count of white blood cells |
|`rbc` | The count of red blood cells |
|`adverse_effects` | The presence of at least a single adverse effect |
|`num_effects` | The number of adverse effects experienced by a single individual |

The original dataset can be found [here](https://hbiostat.org/data/repo/safety.rda).

Your organization has asked you to explore and answer some questions from the data collected. See the project instructions.

In [1]:
# Import packages
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import pingouin
import seaborn as sns
import matplotlib.pyplot as plt

# Load the dataset
drug_safety = pd.read_csv("drug_safety.csv")

# Count of adverse effects and total observations per group
adverse_counts = (
    drug_safety.groupby('trx')['adverse_effects']
    .value_counts()
    .unstack(fill_value=0)
)
# Number who experienced adverse effects (Yes) per group
drug_yes    = adverse_counts.loc["Drug", "Yes"]
placebo_yes = adverse_counts.loc["Placebo", "Yes"]

# Total observations per group
drug_total    = adverse_counts.loc["Drug"].sum()
placebo_total = adverse_counts.loc["Placebo"].sum()

count = np.array([drug_yes, placebo_yes])
nobs  = np.array([drug_total, placebo_total])

stat, two_sample_p_value = proportions_ztest(count, nobs)
print(f"two_sample_p_value: {two_sample_p_value:.4f}")


num_effects_table, _, stats = pingouin.chi2_independence(
    data=drug_safety, x="trx", y="num_effects"
)

num_effects_p_value = stats["pval"][0]
print(f"num_effects_p_value: {num_effects_p_value:.4f}")

drug_ages    = drug_safety.loc[drug_safety["trx"] == "Drug",    "age"]
placebo_ages = drug_safety.loc[drug_safety["trx"] == "Placebo", "age"]

mwu_results = pingouin.mwu(drug_ages, placebo_ages, alternative="two-sided")

age_group_effects_p_value = mwu_results["p-val"].values[0]
print(f"age_group_effects_p_value: {age_group_effects_p_value:.4f}")


two_sample_p_value: 0.9639
num_effects_p_value: 0.6150
age_group_effects_p_value: 0.2570
